In [1]:
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.9.0+cu126
Torch CUDA: 12.6


In [2]:
pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.5.4/flash_attn-2.6.3+cu124torch2.9-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.7/182.7 MB 13.3 MB/s eta 0:00:00


In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Flash SDP Enabled: {torch.backends.cuda.flash_sdp_enabled()}")  # True
import flash_attn
print(f"Flash Version: {flash_attn.__version__}")  # 2.6.3

PyTorch: 2.9.0+cu126
CUDA: 12.6
Flash SDP Enabled: True
Flash Version: 2.6.3


In [4]:
# Check GPU
!nvidia-smi

# Check if we're on Colab
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"Running on Colab: {IN_COLAB}")

Sun Dec  7 16:01:35 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
# Clone repository (if on Colab)
import os

REPO_URL = "https://github.com/Pkansagra-hub/Family_osModernBERT.git"
REPO_DIR = "Modeling_studio"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print("📥 Cloning repository...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("📂 Repository already exists, pulling latest...")
        !cd {REPO_DIR} && git pull

    os.chdir(REPO_DIR)
    print(f"📁 Working directory: {os.getcwd()}")
else:
    # Local development - assume we're in the repo root
    print(f"📁 Working directory: {os.getcwd()}")

📥 Cloning repository...
Cloning into 'Modeling_studio'...
remote: Enumerating objects: 2325, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 2325 (delta 132), reused 176 (delta 92), pack-reused 2084 (from 2)
Receiving objects: 100% (2325/2325), 188.24 MiB | 31.78 MiB/s, done.
Resolving deltas: 100% (1436/1436), done.
Updating files: 100% (595/595), done.
📁 Working directory: /content/Modeling_studio


In [6]:
# Install dependencies
print("📦 Installing dependencies...")
!pip install -q -e .
!pip install -q wandb tensorboard

print("✅ Dependencies installed!")

📦 Installing dependencies...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Building editable for modeling-studio (pyproject.toml) ... done
✅ Dependencies installed!


In [7]:
!pip install -e . 'datasets>=2.14.0,<3.0.0' -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 16.2 MB/s eta 0:00:00
  Building editable for modeling-studio (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [8]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create symlinks for data and outputs
    DRIVE_BASE = "/content/drive/MyDrive/FamilyOS_ModernBERT"
    !mkdir -p "{DRIVE_BASE}/data"
    !mkdir -p "{DRIVE_BASE}/outputs"
    !mkdir -p "{DRIVE_BASE}/checkpoints"

    # Symlink outputs to Drive for persistence
    !rm -rf outputs checkpoints 2>/dev/null
    !ln -s "{DRIVE_BASE}/outputs" outputs
    !ln -s "{DRIVE_BASE}/checkpoints" checkpoints

    print(f"📂 Data/outputs will be saved to: {DRIVE_BASE}")

Mounted at /content/drive
📂 Data/outputs will be saved to: /content/drive/MyDrive/FamilyOS_ModernBERT


In [9]:
!export TF_CPP_MIN_LOG_LEVEL=2  # Hides INFO/WARNING, keeps ERROR
!export XLA_FLAGS=--xla_gpu_cuda_data_dir=/usr/local/cuda  # Points XLA to CUDA (if not auto-detected)

In [10]:
import os
from pathlib import Path

def check_data_exists():
    """Verify required data directories exist."""
    required_paths = {
        "Stage A - NER": "data/public/civil_comments_curated",
        "Stage A - Emotions": "data/familyos/emotions/silver",
        "Stage A - Temporal": "data/familyos/temporal/silver",
    }

    optional_paths = {
        "Stage B - Unified Synthetic": "data/familyos/unified/output_synthetic",
        "Stage B - Unified": "data/familyos/unified/output",
    }

    print("📂 Checking required data...")
    all_ok = True

    for name, path in required_paths.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"   {status} {name}: {path}")
        if not exists:
            all_ok = False

    print("\n📂 Checking optional data (Stage B)...")
    stage_b_ok = False
    for name, path in optional_paths.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "⚠️"
        print(f"   {status} {name}: {path}")
        if exists:
            stage_b_ok = True

    return all_ok, stage_b_ok

stage_a_ready, stage_b_ready = check_data_exists()

if not stage_a_ready:
    print("\n❌ Stage A data missing! Please upload data or mount Google Drive.")
elif not stage_b_ready:
    print("\n⚠️ Stage B data missing - will skip Stage B training.")
else:
    print("\n✅ All data ready!")

📂 Checking required data...
   ✅ Stage A - NER: data/public/civil_comments_curated
   ✅ Stage A - Emotions: data/familyos/emotions/silver
   ✅ Stage A - Temporal: data/familyos/temporal/silver

📂 Checking optional data (Stage B)...
   ✅ Stage B - Unified Synthetic: data/familyos/unified/output_synthetic
   ✅ Stage B - Unified: data/familyos/unified/output

✅ All data ready!


In [ ]:
!cd /content/Modeling_studio && git pull origin main

In [ ]:
%%time

import os


# Use shell command with ! to stream output in real-time
!python scripts/train_stage_a.py \
    --config configs/training/multitask/stage_a_a100_fast.yaml

# Check if training succeeded
if os.path.exists("outputs/modernbert-multitask-v0-stage-a-fast/pytorch_model.bin"):
    print("\n" + "="*60)
    print("✅ STAGE A COMPLETED SUCCESSFULLY!")
    print("="*60)
else:
    print("\n" + "="*60)
    print("❌ STAGE A FAILED! Check logs above.")
    print("="*60)

In [ ]:
# Copy final model to Drive (run AFTER training completes)
!cp -r outputs/ /content/drive/MyDrive/modernbert-multitask-v0-stage-a-fast

In [11]:
# Verify Stage A output
import os
import json

stage_a_output = "outputs/modernbert-multitask-v0-stage-a-fast"

if os.path.exists(stage_a_output):
    print(f"📁 Stage A output: {stage_a_output}")
    print("   Files:")
    for f in os.listdir(stage_a_output):
        size = os.path.getsize(os.path.join(stage_a_output, f)) / 1e6
        print(f"      {f} ({size:.1f} MB)")

    # Load eval results
    eval_path = os.path.join(stage_a_output, "eval_results.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            results = json.load(f)
        print("\n📊 Stage A Eval Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"❌ Stage A output not found at {stage_a_output}")

📁 Stage A output: outputs/modernbert-multitask-v0-stage-a-fast
   Files:
      config.json (0.0 MB)
      pytorch_model.bin (308.9 MB)
      capabilities.json (0.0 MB)
      tokenizer_config.json (0.0 MB)
      special_tokens_map.json (0.0 MB)
      tokenizer.json (3.6 MB)
      training_config.json (0.0 MB)
      training_args.json (0.0 MB)
      eval_results.json (0.0 MB)

📊 Stage A Eval Results:
      epoch: 4.0000
      eval_avg_f1: 0.5564
      eval_avg_score: 0.4079
      eval_best_score: 0.7933
      eval_embedding_loss: 0.3782
      eval_embedding_pearson: 0.0776
      eval_embedding_spearman: 0.1069
      eval_emotions_accuracy: 0.5300
      eval_emotions_f1: 0.4732
      eval_emotions_loss: 1.2918
      eval_emotions_macro_f1: 0.3271
      eval_emotions_precision: 0.5232
      eval_emotions_recall: 0.5300
      eval_ner_general_LOC_f1: 0.6082
      eval_ner_general_MISC_f1: 0.3723
      eval_ner_general_ORG_f1: 0.3250
      eval_ner_general_PER_f1: 0.6115
      eval_ner_gener

In [12]:
!cd /content/Modeling_studio && git pull origin main

From https://github.com/Pkansagra-hub/Family_osModernBERT
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
%%time

import os

# Use shell command with ! to stream output in real-time
!python scripts/train_stage_b.py \
        --config configs/training/multitask/stage_b_for_v3_prep.yaml

# Check if training succeeded
if os.path.exists("outputs/modernbert-v2-for-v3-transfer/pytorch_model.bin"):
  print("\n" + "="*60)
  print("✅ STAGE B COMPLETED SUCCESSFULLY!")
  print("="*60)
else:
  print("\n" + "="*60)
  print("❌ STAGE B FAILED! Check logs above.")
  print("="*60)

2025-12-07 16:05:03.834221: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-07 16:05:03.850837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765123503.871890    2707 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765123503.878259    2707 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765123503.894513    2707 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# Verify Stage B output
import os
import json

stage_b_output = "outputs/modernbert-v2-for-v3-transfer"

if os.path.exists(stage_b_output):
    print(f"📁 Stage B output: {stage_b_output}")
    print("   Files:")
    for f in os.listdir(stage_b_output):
        size = os.path.getsize(os.path.join(stage_b_output, f)) / 1e6
        print(f"      {f} ({size:.1f} MB)")

    # Load eval results
    eval_path = os.path.join(stage_b_output, "eval_results.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            results = json.load(f)
        print("\n📊 Stage B Eval Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"ℹ️ Stage B output not found (may have been skipped)")

ℹ️ Stage B output not found (may have been skipped)


In [ ]:
import os
from datetime import datetime

print("="*60)
print("📋 TRAINING PIPELINE SUMMARY")
print("="*60)
print(f"   Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check outputs
stage_a_ok = os.path.exists("outputs/modernbert-multitask-v0-stage-a-fast/pytorch_model.bin")
stage_b_ok = os.path.exists("outputs/modernbert-v2-for-v3-transfer/pytorch_model.bin")

print(f"\n   Stage A: {'✅ Complete' if stage_a_ok else '❌ Failed/Skipped'}")
print(f"   Stage B: {'✅ Complete' if stage_b_ok else '⚠️ Skipped (no FamilyOS data)'}")

print("\n" + "="*60)
print("📦 OUTPUT CHECKPOINTS")
print("="*60)

if stage_a_ok:
    print("   Stage A: outputs/modernbert-multitask-v0-stage-a-fast")
    print("            → Generic multi-task encoder (7 heads)")

if stage_b_ok:
    print("   Stage B: outputs/modernbert-v2-for-v3-transfer")
    print("            → FamilyOS-tuned encoder for v3 weight transfer")
    print("            → Layers 15-20 trained on family context")

print("\n" + "="*60)
print("🔜 NEXT STEPS")
print("="*60)
print("   1. Download checkpoints from Google Drive")
print("   2. Run v3 initialization script to:")
print("      - Copy v2 layers 1-22 → v3 layers 1-22")
print("      - Clone v2 layers 15-20 → v3 layers 23-28")
print("      - Initialize new v3 heads + hub tokens")
print("   3. Train v3 on FamilyOS data with new architecture")
print("="*60)

📋 TRAINING PIPELINE SUMMARY
   Completed at: 2025-12-07 07:35:57

   Stage A: ✅ Complete
   Stage B: ⚠️ Skipped (no FamilyOS data)

📦 OUTPUT CHECKPOINTS
   Stage A: outputs/modernbert-multitask-v0-stage-a-fast
            → Generic multi-task encoder (7 heads)

🔜 NEXT STEPS
   1. Download checkpoints from Google Drive
   2. Run v3 initialization script to:
      - Copy v2 layers 1-22 → v3 layers 1-22
      - Clone v2 layers 15-20 → v3 layers 23-28
      - Initialize new v3 heads + hub tokens
   3. Train v3 on FamilyOS data with new architecture


In [ ]:
# Optional: Copy outputs to a specific Google Drive location
if IN_COLAB:
    import shutil
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    backup_dir = f"/content/drive/MyDrive/FamilyOS_ModernBERT/runs/{timestamp}"

    print(f"📦 Creating backup at: {backup_dir}")
    os.makedirs(backup_dir, exist_ok=True)

    # Copy Stage A
    if os.path.exists("outputs/modernbert-multitask-v0-stage-a-fast"):
        shutil.copytree(
            "outputs/modernbert-multitask-v0-stage-a-fast",
            f"{backup_dir}/stage_a",
            dirs_exist_ok=True
        )
        print("   ✅ Stage A backed up")

    # Copy Stage B
    if os.path.exists("outputs/modernbert-v2-for-v3-transfer"):
        shutil.copytree(
            "outputs/modernbert-v2-for-v3-transfer",
            f"{backup_dir}/stage_b",
            dirs_exist_ok=True
        )
        print("   ✅ Stage B backed up")

    print(f"\n✅ Backup complete: {backup_dir}")